# FastF1 – AML Forecasting Notebook (2025 Season)

Companion notebook to `aml_fastf1_2025.ipynb`. The data preparation pipeline
(class imbalance, augmentation, imputation) lives in the main notebook; this
one focuses on **time-series forecasting** using the 2025 race lap data.

Two targets, both motivated by the project plan:

1. **Race lap-time forecasting** – predict a driver's lap times for the back
   half of a single race given the front half. Showcases the full classical
   stack (naive → ARIMA → SARIMA → SARIMAX → auto-ARIMA) plus a foundation
   model (Amazon Chronos) zero-shot baseline.

2. **Tyre degradation forecasting** – predict how lap times grow within a
   single stint as `TyreLife` increases. Same toolset, narrower scope.

Methods covered (all from the slide deck):

  Naive (last value, mean, drift) · Differencing & stationarity (ADF) ·
  ACF/PACF · ARIMA(p,d,q) · SARIMA(p,d,q)(P,D,Q,m) · SARIMAX (exogenous) ·
  auto-ARIMA · Chronos foundation model · residual analysis (Q-Q, Ljung-Box)

Group 6: Aritz Ryan San Sebastian · Nico Azcarate · Jon Larrañaga

---
## 0. Setup

In [ ]:
import os, sys, gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

DATA_DIR = "data"
OUT_FC   = "outputs/forecasting"
os.makedirs(OUT_FC, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Library availability (graceful if any are missing) ──────────────
HAS_STATSMODELS = HAS_PMDARIMA = HAS_CHRONOS = False
try:
    import statsmodels.api as sm
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    from statsmodels.tsa.seasonal import STL
    from statsmodels.tsa.stattools import adfuller, acf, pacf
    from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
    from statsmodels.stats.diagnostic import acorr_ljungbox
    HAS_STATSMODELS = True
except ImportError:
    print("[!] statsmodels missing — install with: pip install statsmodels")

try:
    import pmdarima as pm
    HAS_PMDARIMA = True
except ImportError:
    print("[!] pmdarima missing — install with: pip install pmdarima")

try:
    import torch
    from chronos import BaseChronosPipeline
    HAS_CHRONOS = True
except ImportError:
    print("[!] chronos-forecasting missing — install with: pip install chronos-forecasting")

print(f"\nLibraries: statsmodels={HAS_STATSMODELS}  "
      f"pmdarima={HAS_PMDARIMA}  chronos={HAS_CHRONOS}")

---
## 1. Data Loading

Loads `data/laps.csv` (already produced by the main notebook). Each row is a
single lap with weather merged in by nearest session time.

In [ ]:
laps = pd.read_csv(f"{DATA_DIR}/laps.csv")
print(f"Loaded {len(laps):,} laps across {laps['Round'].nunique()} rounds, "
      f"{laps['Driver'].nunique()} drivers.")

# Normalise dtypes that round-trip badly through CSV (same fix as main notebook)
if "TrackStatus" in laps.columns:
    laps["TrackStatus"] = laps["TrackStatus"].astype(str)
if "Deleted" in laps.columns:
    laps["Deleted"] = (
        laps["Deleted"].astype(str).str.strip().str.lower()
        .isin(["true", "1", "1.0"])
    )

# LapTime/Sector*Time were saved as float seconds — confirm
print(f"\nLap-time stats (seconds):")
print(laps["LapTime"].describe().round(2).to_string())

### Shared evaluation helper

Used in both forecasting sections so the metric reporting is consistent.

In [ ]:
def forecast_metrics(y_true, y_pred):
    """MAE, RMSE, MAPE.  All inputs 1-D numeric arrays of equal length."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask   = ~(np.isnan(y_true) | np.isnan(y_pred))
    yt, yp = y_true[mask], y_pred[mask]
    if len(yt) == 0:
        return dict(MAE=np.nan, RMSE=np.nan, MAPE=np.nan)
    mae  = float(np.mean(np.abs(yt - yp)))
    rmse = float(np.sqrt(np.mean((yt - yp) ** 2)))
    nz   = yt != 0
    mape = float(np.mean(np.abs((yt[nz] - yp[nz]) / yt[nz])) * 100) if nz.any() else np.nan
    return dict(MAE=mae, RMSE=rmse, MAPE=mape)

def fmt_metrics(name, m):
    return f"    {name:<22s}  MAE={m['MAE']:.3f}s  RMSE={m['RMSE']:.3f}s  MAPE={m['MAPE']:.2f}%" 

---
## 2. Race Lap-Time Forecasting

Forecast a single driver's lap times across one race. Train on the first ~70 %
of the race; test on the last ~30 %.

### 2.1 Choose subject (driver + race)

We pick the driver-race combination with the most laps after dropping pit
in/out laps and rows missing `LapTime` — that gives the longest clean series
for forecasting. The choice is parametric: change `SUBJECT_DRIVER` /
`SUBJECT_ROUND` below to override.

In [ ]:
# Override these to pin a specific subject; otherwise auto-pick the longest
# clean series.
SUBJECT_DRIVER = None       # e.g. "VER"
SUBJECT_ROUND  = None       # e.g. 1

# Drop pit-in / pit-out / NaN-LapTime laps which break time-series continuity
clean = laps.dropna(subset=["LapTime"]).copy()
clean = clean[clean["PitOutTime"].isna() & clean["PitInTime"].isna()]

if SUBJECT_DRIVER is None or SUBJECT_ROUND is None:
    counts = (clean.groupby(["Round", "Driver"]).size()
                  .sort_values(ascending=False))
    (best_round, best_driver) = counts.index[0]
    SUBJECT_ROUND  = SUBJECT_ROUND  or int(best_round)
    SUBJECT_DRIVER = SUBJECT_DRIVER or str(best_driver)

subj = clean[(clean["Round"] == SUBJECT_ROUND) &
             (clean["Driver"] == SUBJECT_DRIVER)].copy()
subj = subj.sort_values("LapNumber").reset_index(drop=True)

event_name = subj["EventName"].iloc[0] if "EventName" in subj.columns else f"Round {SUBJECT_ROUND}"
print(f"Subject: {SUBJECT_DRIVER} @ {event_name} (Round {SUBJECT_ROUND})")
print(f"Clean laps: {len(subj)}")
print(f"Stints in race: {sorted(subj['Stint'].dropna().unique().tolist())}")
print(f"Compounds used: {subj['Compound'].dropna().unique().tolist()}")

### 2.2 Visualisation

Lap time as a function of `LapNumber`, coloured by stint.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for stint, g in subj.groupby("Stint"):
    compound = g["Compound"].iloc[0] if not g["Compound"].isna().all() else "?"
    ax.plot(g["LapNumber"], g["LapTime"], "o-",
            label=f"Stint {int(stint)} ({compound})", lw=1.5, ms=4)
ax.set_xlabel("Lap number"); ax.set_ylabel("Lap time (s)")
ax.set_title(f"{SUBJECT_DRIVER} @ {event_name} — lap times by stint")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### 2.3 STL decomposition

Splits the series into trend, seasonal (period = median stint length), and
residual components. The trend captures fuel-burn + track evolution; the
seasonal component captures repeating tyre-degradation cycles per stint.

In [ ]:
ts = subj.set_index("LapNumber")["LapTime"].astype(float)

if HAS_STATSMODELS and len(ts) >= 14:
    # Period = median stint length, clamped so STL has at least 2 cycles
    stint_lens = subj.groupby("Stint").size()
    period = max(2, int(min(stint_lens.median(), len(ts) // 2)))
    print(f"STL period = {period} (median stint length)")

    stl = STL(ts.values, period=period, robust=True).fit()
    fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
    axes[0].plot(ts.index, ts.values);          axes[0].set_title("Observed lap times (s)")
    axes[1].plot(ts.index, stl.trend);          axes[1].set_title("Trend (fuel burn + track evolution)")
    axes[2].plot(ts.index, stl.seasonal);       axes[2].set_title(f"Seasonal (period={period})")
    axes[3].plot(ts.index, stl.resid);          axes[3].set_title("Residual")
    for ax in axes: ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("Skipping STL — statsmodels unavailable or series too short.")

### 2.4 Stationarity test (ADF)

If the p-value is below 0.05 we reject the unit-root null and treat the
series as stationary. Otherwise we'll need to difference.

In [ ]:
if HAS_STATSMODELS:
    stat, pval, *_ = adfuller(ts.dropna().values)
    print(f"ADF test on raw lap times:  stat={stat:.3f}  p-value={pval:.4f}")
    print(f"  -> {'stationary (reject H0)' if pval < 0.05 else 'NOT stationary (cannot reject H0)'}")

    # First difference
    ts_d1 = ts.diff().dropna()
    stat1, pval1, *_ = adfuller(ts_d1.values)
    print(f"ADF on first difference:   stat={stat1:.3f}  p-value={pval1:.4f}")
    print(f"  -> {'stationary' if pval1 < 0.05 else 'NOT stationary'}")
else:
    print("ADF skipped — statsmodels unavailable.")

### 2.5 Train / test split

Last ~30 % of laps held out for testing. The training window is what every
forecaster sees; the test window is what we score against.

In [ ]:
n = len(ts)
TEST_FRAC = 0.30
n_test = max(5, int(n * TEST_FRAC))
n_train = n - n_test

ts_train = ts.iloc[:n_train]
ts_test  = ts.iloc[n_train:]
print(f"Train: {n_train} laps  |  Test: {n_test} laps  |  Horizon: {n_test}")

### 2.6 Naive baselines

Three flavours straight from the slides:

* **Last value** — repeat `ts_train.iloc[-1]` for the whole horizon.
* **Mean** — repeat `ts_train.mean()`.
* **Drift** — linear extrapolation through first and last train points.

In [ ]:
results = {}     # method -> dict of metrics
forecasts = {}   # method -> ndarray prediction over the test horizon

# 1) Last-value naive
last_val = float(ts_train.iloc[-1])
forecasts["Naive (last)"] = np.full(n_test, last_val)

# 2) Mean
forecasts["Naive (mean)"] = np.full(n_test, float(ts_train.mean()))

# 3) Drift
slope = (ts_train.iloc[-1] - ts_train.iloc[0]) / max(len(ts_train) - 1, 1)
forecasts["Naive (drift)"] = ts_train.iloc[-1] + slope * np.arange(1, n_test + 1)

for name, f in forecasts.items():
    results[name] = forecast_metrics(ts_test.values, f)
    print(fmt_metrics(name, results[name]))

### 2.7 Differencing & ACF / PACF

ACF and PACF on the differenced series tell us starting points for `q` and
`p` respectively. Significant lags = bars outside the blue confidence bands.

In [ ]:
if HAS_STATSMODELS:
    ts_train_d1 = ts_train.diff().dropna()
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    plot_acf(ts_train_d1, lags=min(20, len(ts_train_d1) // 2 - 1), ax=axes[0])
    axes[0].set_title("ACF of first-differenced lap times (suggests q)")
    plot_pacf(ts_train_d1, lags=min(20, len(ts_train_d1) // 2 - 1), ax=axes[1], method="ywm")
    axes[1].set_title("PACF of first-differenced lap times (suggests p)")
    plt.tight_layout(); plt.show()
else:
    print("Skipped — statsmodels unavailable.")

### 2.8 ARIMA(p,d,q) — manual

We fit a few small candidates and pick by AIC. Then run residual diagnostics
on the winner: Q-Q plot for normality and the Ljung-Box test for
autocorrelation. If the residuals are clean white noise, the model has
captured everything it can.

In [ ]:
arima_winner = None
arima_pred   = None
if HAS_STATSMODELS:
    candidates = [(0, 1, 0), (1, 1, 0), (0, 1, 1), (1, 1, 1), (2, 1, 1), (1, 1, 2), (2, 1, 2)]
    best_aic = np.inf
    best_order = None
    for order in candidates:
        try:
            m = ARIMA(ts_train.values, order=order).fit()
            print(f"  ARIMA{order}  AIC={m.aic:.2f}")
            if m.aic < best_aic:
                best_aic, best_order, arima_winner = m.aic, order, m
        except Exception as e:
            print(f"  ARIMA{order}  FAILED: {e}")
    print(f"\nBest ARIMA order: {best_order}  (AIC={best_aic:.2f})")

    arima_pred = arima_winner.forecast(steps=n_test)
    forecasts[f"ARIMA{best_order}"] = np.asarray(arima_pred)
    results[f"ARIMA{best_order}"]   = forecast_metrics(ts_test.values, arima_pred)
    print(fmt_metrics(f"ARIMA{best_order}", results[f"ARIMA{best_order}"]))

In [ ]:
# Residual diagnostics for the winning ARIMA
if arima_winner is not None:
    resid = arima_winner.resid[1:]   # drop first (uninformative)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    sm.qqplot(resid, line="45", ax=axes[0])
    axes[0].set_title("Q-Q plot of residuals (should follow y=x)")

    axes[1].plot(resid); axes[1].axhline(0, color="k", lw=0.5)
    axes[1].set_title("Residuals over time"); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    lb = acorr_ljungbox(resid, lags=[10], return_df=True)
    p  = float(lb["lb_pvalue"].iloc[0])
    print(f"Ljung-Box (lag=10): p-value={p:.4f}  "
          f"-> {'residuals look like white noise' if p > 0.05 else 'residuals still autocorrelated'}")

### 2.9 SARIMA(p,d,q)(P,D,Q,m) — seasonal

In F1, the natural seasonality is the **stint cycle** — tyres degrade then a
pit stop resets the cycle. We use the median stint length as `m`. Stints
aren't all the same length so this is approximate, but it usually still
helps SARIMA model the periodic tyre-degradation pattern.

In [ ]:
sarima_pred = None
if HAS_STATSMODELS and len(ts_train) >= 30:
    period = max(2, int(min(subj.groupby("Stint").size().median(), len(ts_train) // 3)))
    try:
        sarima = SARIMAX(
            ts_train.values, order=(1, 1, 1), seasonal_order=(1, 1, 1, period),
            enforce_stationarity=False, enforce_invertibility=False,
        ).fit(disp=False)
        sarima_pred = sarima.forecast(steps=n_test)
        forecasts[f"SARIMA(1,1,1)(1,1,1,{period})"] = np.asarray(sarima_pred)
        results[f"SARIMA(1,1,1)(1,1,1,{period})"]   = forecast_metrics(ts_test.values, sarima_pred)
        print(fmt_metrics(f"SARIMA(1,1,1)(1,1,1,{period})",
                          results[f"SARIMA(1,1,1)(1,1,1,{period})"]))
        print(f"AIC: {sarima.aic:.2f}")
    except Exception as e:
        print(f"SARIMA failed: {e}")
else:
    print("SARIMA skipped — series too short or statsmodels unavailable.")

### 2.10 SARIMAX — exogenous variables

This is where domain knowledge enters. `TyreLife`, `Stint`, `TrackTemp`,
`AirTemp`, `Humidity`, and one-hot `Compound` give the model context that
helps it explain why the next lap will be slow or fast — pit stops, tyre
choice, weather. We have to forecast the exogenous values for the test
horizon (we use them directly since they're observed; in a true forecasting
deployment we'd predict them too).

In [ ]:
sarimax_pred = None
EXOG_COLS = ["TyreLife", "Stint", "TrackTemp", "AirTemp", "Humidity"]
exog_cols_present = [c for c in EXOG_COLS if c in subj.columns]

# Build exogenous matrix aligned with ts (same length, same order)
exog_df = subj.set_index("LapNumber")[exog_cols_present].astype(float)

# One-hot Compound, joined onto exog
if "Compound" in subj.columns:
    cmp_oh = pd.get_dummies(subj.set_index("LapNumber")["Compound"],
                             prefix="Cmp", dtype=float)
    exog_df = exog_df.join(cmp_oh)

# Reindex to match ts and fill any gaps with forward/back fill
exog_df = exog_df.reindex(ts.index).ffill().bfill()

exog_train = exog_df.iloc[:n_train].values
exog_test  = exog_df.iloc[n_train:].values

if HAS_STATSMODELS and len(ts_train) >= 30:
    try:
        sarimax = SARIMAX(
            ts_train.values, exog=exog_train,
            order=(1, 1, 1), seasonal_order=(1, 0, 1, period if 'period' in dir() else 5),
            enforce_stationarity=False, enforce_invertibility=False,
        ).fit(disp=False)
        sarimax_pred = sarimax.forecast(steps=n_test, exog=exog_test)
        forecasts["SARIMAX (+exog)"] = np.asarray(sarimax_pred)
        results["SARIMAX (+exog)"]   = forecast_metrics(ts_test.values, sarimax_pred)
        print(fmt_metrics("SARIMAX (+exog)", results["SARIMAX (+exog)"]))
        print(f"AIC: {sarimax.aic:.2f}")
        print(f"Exogenous columns: {list(exog_df.columns)}")
    except Exception as e:
        print(f"SARIMAX failed: {e}")
else:
    print("SARIMAX skipped.")

### 2.11 auto-ARIMA

`pmdarima.auto_arima` does the heavy lifting — it grid-searches over `(p,d,q)`
(and seasonal orders if `seasonal=True`) using AIC. We let it find its own
SARIMAX too by passing the exogenous matrix.

In [ ]:
auto_pred = None
if HAS_PMDARIMA:
    try:
        auto = pm.auto_arima(
            ts_train.values, exogenous=exog_train,
            seasonal=True, m=max(2, int(subj.groupby('Stint').size().median())),
            stepwise=True, suppress_warnings=True,
            error_action="ignore", trace=False,
            max_p=3, max_q=3, max_P=1, max_Q=1, max_d=2, max_D=1,
        )
        auto_pred = auto.predict(n_periods=n_test, exogenous=exog_test)
        forecasts["auto-ARIMA"] = np.asarray(auto_pred)
        results["auto-ARIMA"]   = forecast_metrics(ts_test.values, auto_pred)
        print(f"auto-ARIMA chose: {auto.order}  seasonal={auto.seasonal_order}  AIC={auto.aic():.2f}")
        print(fmt_metrics("auto-ARIMA", results["auto-ARIMA"]))
    except Exception as e:
        print(f"auto-ARIMA failed: {e}")
else:
    print("auto-ARIMA skipped — pmdarima unavailable.")

### 2.12 Chronos — foundation-model zero-shot

Chronos is a pre-trained time-series foundation model (Amazon, 2024). It's
**zero-shot**: no fitting, no hyperparameter search — we just hand it the
training history and ask for a forecast. We use `chronos-bolt-small` (the
fastest variant; ~50 M parameters). First call downloads ~120 MB of weights.

In [ ]:
chronos_pred = None
if HAS_CHRONOS:
    try:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Loading Chronos-Bolt-small on {device} ...")
        pipe = BaseChronosPipeline.from_pretrained(
            "amazon/chronos-bolt-small",
            device_map=device,
            torch_dtype=torch.float32,
        )
        ctx = torch.tensor(ts_train.values, dtype=torch.float32)
        # Chronos-Bolt returns (batch, prediction_length, n_quantiles)
        quantiles, mean = pipe.predict_quantiles(
            context=ctx, prediction_length=n_test,
            quantile_levels=[0.1, 0.5, 0.9],
        )
        chronos_pred = quantiles[0, :, 1].cpu().numpy()  # median (q=0.5)
        forecasts["Chronos (zero-shot)"] = chronos_pred
        results["Chronos (zero-shot)"]   = forecast_metrics(ts_test.values, chronos_pred)
        print(fmt_metrics("Chronos (zero-shot)", results["Chronos (zero-shot)"]))
    except Exception as e:
        print(f"Chronos failed: {e}")
else:
    print("Chronos skipped — chronos-forecasting unavailable.")

### 2.13 Comparison

Sorted by MAE. Lower is better. Look for any method that beats the naive
baselines on **all three** metrics — that's a real lift.

In [ ]:
metrics_df = pd.DataFrame(results).T.sort_values("MAE")
print("\n── Race lap-time forecasting results ──────────────────────────")
print(metrics_df.round(3).to_string())
metrics_df.to_csv(f"{OUT_FC}/race_laptime_metrics.csv")

# Plot all forecasts on one chart
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(ts.index, ts.values, "k-", lw=2, label="Actual", alpha=0.8)
ax.axvline(ts_train.index[-1] + 0.5, color="gray", ls="--", label="train/test split")

colors = plt.cm.tab10(np.linspace(0, 1, len(forecasts)))
for (name, f), c in zip(forecasts.items(), colors):
    ax.plot(ts_test.index, f, "o-", color=c, lw=1.5, ms=4, label=name, alpha=0.85)

ax.set_xlabel("Lap number"); ax.set_ylabel("Lap time (s)")
ax.set_title(f"All forecasts vs actual — {SUBJECT_DRIVER} @ {event_name}")
ax.legend(loc="best", fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{OUT_FC}/race_laptime_forecasts.png", dpi=120); plt.show()

---
## 3. Tyre Degradation Forecasting

Per-stint forecasting: given the first half of a stint (where the driver is
typically still on fresh-ish rubber) predict the second half (where
degradation kicks in). The exogenous lever is `TyreLife`.

### 3.1 Stint extraction

Group by `(Round, Driver, Stint)` and keep stints with ≥10 clean laps so
forecasting actually has something to learn from.

In [ ]:
STINT_MIN_LAPS = 10
stint_groups = (clean.groupby(["Round", "Driver", "Stint"])
                     .size().reset_index(name="laps")
                     .query("laps >= @STINT_MIN_LAPS"))
print(f"Eligible stints (>= {STINT_MIN_LAPS} laps): {len(stint_groups)}")
print(stint_groups.head(10).to_string(index=False))

### 3.2 Visualise stint lap-times vs TyreLife

Each line is one stint. The cliff (sudden lap-time spike) is what we'd love
the forecaster to predict before it happens.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
plotted = 0
for (rnd, drv, stnt), g in clean.groupby(["Round", "Driver", "Stint"]):
    if len(g) < STINT_MIN_LAPS:  continue
    g = g.sort_values("LapNumber")
    if "TyreLife" not in g.columns or g["TyreLife"].isna().all():  continue
    if plotted >= 25:  break  # don't overplot
    ax.plot(g["TyreLife"], g["LapTime"], "-", alpha=0.4, lw=1)
    plotted += 1
ax.set_xlabel("Tyre life (laps)"); ax.set_ylabel("Lap time (s)")
ax.set_title(f"Lap time vs tyre life ({plotted} sample stints)")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### 3.3 Forecast on the longest available stint

Same train/test approach: first 70 % of stint laps to fit, last 30 % to
score. Apply the methods that fit best for short single-stint series:

* Naive (last + drift)
* ARIMA (small order)
* SARIMAX with `TyreLife` as the only exogenous variable
* Chronos zero-shot

In [ ]:
# Pick the longest stint
longest = stint_groups.sort_values("laps", ascending=False).iloc[0]
S_RND, S_DRV, S_STNT = int(longest["Round"]), str(longest["Driver"]), int(longest["Stint"])
stint = (clean[(clean["Round"] == S_RND) &
               (clean["Driver"] == S_DRV) &
               (clean["Stint"] == S_STNT)]
         .sort_values("LapNumber").reset_index(drop=True))
print(f"Longest stint: Round {S_RND}, {S_DRV}, Stint {S_STNT}, "
      f"{len(stint)} laps, compound={stint['Compound'].iloc[0]}")

ts_s = stint.set_index("LapNumber")["LapTime"].astype(float)
n2 = len(ts_s)
n2_test  = max(3, int(n2 * 0.30))
n2_train = n2 - n2_test
ts_s_train, ts_s_test = ts_s.iloc[:n2_train], ts_s.iloc[n2_train:]
exog_s = stint.set_index("LapNumber")[["TyreLife"]].astype(float).reindex(ts_s.index).ffill().bfill()
exog_s_train = exog_s.iloc[:n2_train].values
exog_s_test  = exog_s.iloc[n2_train:].values

deg_results, deg_forecasts = {}, {}

# Naive last
deg_forecasts["Naive (last)"] = np.full(n2_test, float(ts_s_train.iloc[-1]))
# Drift
slope_s = (ts_s_train.iloc[-1] - ts_s_train.iloc[0]) / max(len(ts_s_train) - 1, 1)
deg_forecasts["Naive (drift)"] = ts_s_train.iloc[-1] + slope_s * np.arange(1, n2_test + 1)

# ARIMA
if HAS_STATSMODELS:
    best_aic, best_order, best_model = np.inf, None, None
    for order in [(0,1,0),(1,1,0),(0,1,1),(1,1,1),(2,1,1)]:
        try:
            m = ARIMA(ts_s_train.values, order=order).fit()
            if m.aic < best_aic:
                best_aic, best_order, best_model = m.aic, order, m
        except Exception: pass
    if best_model is not None:
        deg_forecasts[f"ARIMA{best_order}"] = np.asarray(best_model.forecast(steps=n2_test))

# SARIMAX(TyreLife)
if HAS_STATSMODELS and n2_train >= 8:
    try:
        sm_x = SARIMAX(
            ts_s_train.values, exog=exog_s_train, order=(1,1,1),
            enforce_stationarity=False, enforce_invertibility=False,
        ).fit(disp=False)
        deg_forecasts["SARIMAX (TyreLife)"] = np.asarray(
            sm_x.forecast(steps=n2_test, exog=exog_s_test)
        )
    except Exception as e:
        print(f"  SARIMAX(TyreLife) failed: {e}")

# Chronos
if HAS_CHRONOS:
    try:
        ctx2 = torch.tensor(ts_s_train.values, dtype=torch.float32)
        q2, _ = pipe.predict_quantiles(  # reuse pipe from section 2
            context=ctx2, prediction_length=n2_test,
            quantile_levels=[0.1, 0.5, 0.9],
        )
        deg_forecasts["Chronos (zero-shot)"] = q2[0, :, 1].cpu().numpy()
    except Exception as e:
        print(f"  Chronos failed: {e}")

for name, f in deg_forecasts.items():
    deg_results[name] = forecast_metrics(ts_s_test.values, f)
    print(fmt_metrics(name, deg_results[name]))

### 3.4 Comparison + plot

In [ ]:
deg_metrics_df = pd.DataFrame(deg_results).T.sort_values("MAE")
print("\n── Tyre degradation forecasting results ──────────────────")
print(deg_metrics_df.round(3).to_string())
deg_metrics_df.to_csv(f"{OUT_FC}/degradation_metrics.csv")

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(ts_s.index, ts_s.values, "k-", lw=2, label="Actual", alpha=0.8)
ax.axvline(ts_s_train.index[-1] + 0.5, color="gray", ls="--", label="train/test split")
colors = plt.cm.tab10(np.linspace(0, 1, max(1, len(deg_forecasts))))
for (name, f), c in zip(deg_forecasts.items(), colors):
    ax.plot(ts_s_test.index, f, "o-", color=c, lw=1.5, ms=4, label=name, alpha=0.85)
ax.set_xlabel("Lap number"); ax.set_ylabel("Lap time (s)")
ax.set_title(f"Tyre degradation forecast — {S_DRV}, Round {S_RND}, Stint {S_STNT} "
             f"({stint['Compound'].iloc[0]})")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{OUT_FC}/degradation_forecasts.png", dpi=120); plt.show()

---
## 4. Summary

What the comparison should typically show on F1 lap-time data:

* **Naive (last)** is a surprisingly strong baseline because lap times
  within a stint are correlated. Beating it requires a model that knows
  about pit stops or tyre age.
* **ARIMA** captures the within-stint autocorrelation but ignores stint
  boundaries — it predicts smoothly through pit stops, missing the
  reset-to-fast-laps after each stop.
* **SARIMA** helps when stint cadence is regular (one or two stops at
  predictable laps).
* **SARIMAX with tyre + weather exogenous** is usually the winner among
  classical methods because it knows when the driver pits and which
  compound goes on.
* **auto-ARIMA** is essentially a guard rail — its order-search converges
  to a similar SARIMAX, sometimes a bit better, sometimes worse.
* **Chronos zero-shot** is the most interesting modern baseline: it has
  never seen F1 data but often produces forecasts comparable to fitted
  classical models. When it loses, it loses on stint-resets (which it
  has no way to know about). When it wins, it wins on smoothness.

For a forecasting deployment, the realistic answer is "SARIMAX with rich
exogenous features", because pit stops, weather, and tyre choice carry
most of the predictable signal in F1 lap times.

### Files written

* `outputs/forecasting/race_laptime_metrics.csv`
* `outputs/forecasting/race_laptime_forecasts.png`
* `outputs/forecasting/degradation_metrics.csv`
* `outputs/forecasting/degradation_forecasts.png`